In [ ]:
# imports
import numpy as np
import pandas as pd


from collections import defaultdict
from operator import itemgetter
from os import path


In [38]:
# bring in intersections -> DataFrame intersections 
intersection_data_path = "~/code/county_coverage/data/cleaner/intersections_data.json"

# import intersection data
def read_in_intersections(path_to_intersection_json):
    out = pd.read_json(path_to_intersection_json)
    # fix some things on import
    out = out.set_index("INTID")
    out.GEOMETRY = out.GEOMETRY.apply(np.array)
    return out             

intersections = read_in_intersections(path.expanduser(intersection_data_path))

#intersections.GEOMETRY = intersections.GEOMETRY.apply(np.array)
#intersections.GEOMETRY.iloc[0] # == array([-85.51043903,  38.20587931]) # good 

intersections.head()

,FST_ROADNAME,FST_SIFID,SEC_ROADNAME,SEC_SIFID,GEOMETRY
INTID,,,,,
5710837346,REHL RD,4976,W REHL CT,6856,"[-85.5104390301, 38.2058793146]"
10005800273,REHL RD,4976,TUCKER STATION RD,5908,"[-85.52814980550001, 38.2003487994]"
14300767569,REHL RD,4976,TUCKER STATION RD,5908,"[-85.5284139077, 38.2003530575]"
18011691414,I 64 EAST,3076,I 265 RAMP,8763,"[-85.5049493351, 38.2225961436]"
23945910678,I 265 NORTH,8197,I 265 RAMP,8763,"[-85.5055416176, 38.2221199919]"


In [39]:
# bring in centerlines -> DataFrame centerlines
centerlines_path = "~/code/county_coverage/data/cleaner/centerlines_data.json"
centerline_data = pd.read_json(path.expanduser(centerlines_path))

# fix geometry column
split_geo = centerline_data.GEOMETRY.transform({"GEOLOW":itemgetter(0), 'GEOHI':itemgetter(-1)})
centerline_data['GEOLOW'] = split_geo.GEOLOW.apply(np.array)
centerline_data['GEOHI'] = split_geo.GEOHI.apply(np.array)

keep_columns = ["ROADNAME", "SIFID", "SIFIDLOW", "SIFIDHI", "GEOLOW", "GEOHI", "CORE_CLASS"]
centerline_data = centerline_data[keep_columns]
#centerlines.head()
#type(centerlines.GEOLOW.iloc[0])


# remove interstates from centerlines ?\
# - and other roads like ramps?
exclusions = ('EXPRESSWAY', 'INTERSTATE RAMP')
excluded_centerlines = centerline_data[centerline_data.CORE_CLASS.isin(exclusions)].index

centerlines = centerline_data[~centerline_data.CORE_CLASS.isin(exclusions)]

# remove these from intersections as well? 

centerlines.head()

,ROADNAME,SIFID,SIFIDLOW,SIFIDHI,GEOLOW,GEOHI,CORE_CLASS
1,SERENITY CT,8665,1550,8594,"[-85.6809503218, 38.1588670887]","[-85.6812346349, 38.1581804844]",LOCAL
2,S 28TH ST,5926,2854,2470,"[-85.8012012237, 38.230563793]","[-85.8013692698, 38.2293433595]",LOCAL
3,BEECH ST,473,6487,10596,"[-85.8050149882, 38.2289330216]","[-85.8048316249, 38.2275378873]",LOCAL
4,GARDEN DR,2442,13394,4702,"[-85.6802054534, 38.2480671366]","[-85.6798630078, 38.2476081599]",PRIMARY COLLECTOR
5,PARKWAY DR,4573,4162,8594,"[-85.7420397977, 38.2118147709]","[-85.7416475277, 38.2119685968]",LOCAL


In [40]:
def index_by_SIFID_pairs(intersection_sifid_pairs, centerline_sifid_pairs) -> pd.DataFrame:
    return pd.concat((intersection_sifid_pairs, centerline_sifid_pairs), axis=1,
                    # map intersection ids to centerline ids where their sifid pair indexes match
                    # matches are collections of intersection ids/indexes and centerline ids/indexes
                        ).dropna(how='any')
                    # if either collection of ids is empty (or both), there is no match.


\# Old code to match centerlines and intersections by SIFID pairs

```py
def build_2(intersection_sifid_index, centerline_sifid_index, 
                      intersection_dict, centerline_dict) -> None:
    # build intersection sifid pair / centerline sifid pair index
    matchset = index_by_SIFID_pairs(intersection_sifid_index, centerline_sifid_index)

    for _, intersection_ids, centerline_ids in matchset.itertuples():
        for int_id in intersection_ids:
            # gather centerline ids into appropriate dictionaries, indexed by intersection ids
            for cl_id in centerline_ids:
                intersection_dict[(int_id, cl_id)] = True
                centerline_dict[(int_id, cl_id)] = True
```

This code was an improved version of this:

```py
def build_match_dicts(intersection_sifid_index, centerline_sifid_index, 
                      intersection_dict, centerline_dict) -> None:
    # build intersection sifid pair / centerline sifid pair index
    matchset = index_by_SIFID_pairs(intersection_sifid_index, centerline_sifid_index)

    for _, intersection_ids, centerline_ids in matchset.itertuples():
        for int_id in intersection_ids:
            # gather centerline ids into appropriate dictionaries, indexed by intersection ids
            intersection_dict[int_id].update(centerline_ids)
            centerline_dict[int_id].update(centerline_ids)
```

Both required error-prone finagling of data objects like this:

```py
# create dictionaries to store future series info
intersections_FST_match = defaultdict(set)
intersections_SEC_match = defaultdict(set)
centerlines_LOW_match = defaultdict(set)
centerlines_HI_match = defaultdict(set)

intm1 = dict()#pd.Series(name='FST_match')
intm2 = dict()#pd.Series(name='SEC_match')
clLOWm = dict()#pd.Series(name='SIFIDLOW_match')
clHIm = dict()#pd.Series(name='SIFIDHI_match')

build_2(intersections_by_FST_SEC, centerlines_by_SIFID_SIFIDLOW, intm1, clLOWm)
build_2(intersections_by_FST_SEC, centerlines_by_SIFID_SIFIDHI, intm1, clHIm)
build_2(intersections_by_SEC_FST, centerlines_by_SIFID_SIFIDLOW, intm2, clLOWm)
build_2(intersections_by_SEC_FST, centerlines_by_SIFID_SIFIDHI, intm2, clHIm)
# build step takes 30s! # not anymore very fast now for some reaason.
```


```python
# Create indexes, sort information into the appropriate dictionaries.
build_match_dicts(intersections_by_FST_SEC, centerlines_by_SIFID_SIFIDLOW, intersections_FST_match, centerlines_LOW_match)
    # Where intersections[FST_SIFID] == centerlines[SIFID] & intersections[SEC_SIFID] == centerlines[SIFIDLOW]
    # Map intersection ids to the centerline ids. Next, do the same for all the other combinations of columns:
build_match_dicts(intersections_by_FST_SEC, centerlines_by_SIFID_SIFIDHI, intersections_FST_match, centerlines_HI_match)
build_match_dicts(intersections_by_SEC_FST, centerlines_by_SIFID_SIFIDLOW, intersections_SEC_match, centerlines_LOW_match)
build_match_dicts(intersections_by_SEC_FST, centerlines_by_SIFID_SIFIDHI, intersections_SEC_match, centerlines_HI_match)

# Convert the dictionaries to Series.
intersections_FST_match = pd.Series(intersections_FST_match, name='FST_match')
intersections_SEC_match = pd.Series(intersections_SEC_match, name='SEC_match')
centerlines_LOW_match = pd.Series(centerlines_LOW_match, name='SIFIDLOW_match')
centerlines_HI_match = pd.Series(centerlines_HI_match, name='SIFIDHI_match')

# Create dataframe by concatenating Series. Intersection id's are the index. 
matchdf = pd.concat((intersections_FST_match, intersections_SEC_match, centerlines_LOW_match, centerlines_HI_match), axis=1)
matchdf.info()
```

```py
intm1 = pd.Series(intm1, name='FST_match', dtype='boolean')
intm2 = pd.Series(intm2, name='SEC_match', dtype='boolean')
clLOWm = pd.Series(clHIm, name='SIFIDLOW_match', dtype='boolean')
clHIm = pd.Series(clLOWm, name='SIFIDHI_match', dtype='boolean')

newmatch = pd.concat((intm1, intm2, clLOWm, clHIm), axis=1)
newmatch[newmatch.SIFIDHI_match & newmatch.SIFIDLOW_match]
```

In [41]:
# look up roadname by sifid

names = centerlines.groupby("SIFID").ROADNAME.apply(set)
all(names.apply(len) == 1) # -> True. :)
sifid_to_roadname = names.apply(set.pop)

sifid_to_roadname



SIFID
1                    NO NAME
2                   ABBEY RD
3               ABBEYWOOD RD
4           ABBOTTS BEACH RD
5                  ABELL AVE
                ...         
15638    HURSTBOURNE VIEW DR
15639            MEGAWATT DR
15640           GREY WOLF DR
15641        MEADOWSTONE TRL
15642               ASHER CT
Name: ROADNAME, Length: 11330, dtype: object

In [42]:
# get groups
def get_groups(df, by) -> dict:
    return pd.Series(df.groupby(by=by).groups)

centerlines_by_SIFID_SIFIDLOW = get_groups(centerlines, ['SIFID', 'SIFIDLOW'])
centerlines_by_SIFID_SIFIDHI = get_groups(centerlines, ['SIFID', 'SIFIDHI'])

intersections_by_FST_SEC = get_groups(intersections, ['FST_SIFID', 'SEC_SIFID'])
# If the index of one of the centerline groups matches an index in this group, that means
#
#   (centerline == centerlines.loc[centerline index])
#   intersection[FST_SIFID] == centerline[SIFID]
#                   and
#   intersections[SEC_SIFID] == either centerline[SIFIDLOW] or centerline[SIFIDHI]
# depending on which set of centerline groups, which we built above, that we are using. 

# The names FST_SIFID / SEC_SIFID -> First / Second intersection SIFID are convention and
# the order is arbitrary. Reversing the order of the index allows us to match SEC_SIFID efficiently.
intersections_by_SEC_FST = get_groups(intersections, ['SEC_SIFID', 'FST_SIFID'])
# If the index of one of the centerline groups is an index in this group, that means
#
#   intersection[SEC_SIFID] == centerline[SIFID]
#                   and
#   intersections[FST_SIFID] == either centerline[SIFIDLOW] or centerline[SIFIDHI]


In [43]:
# old version of the next cell.
# some code still depends on this cell, but functionally, the next cell does the same thing more efficiently

mapping = defaultdict(int)

start_value = 0b1000 # set start_value flag for first iteration over itersections_by_FST_SEC
for I in (intersections_by_FST_SEC, intersections_by_SEC_FST):

    cvalue = 0b10 # set cvalue flag for first iteration over centerlines_by_SIFID_SIFIDLOW
    for C in (centerlines_by_SIFID_SIFIDLOW, centerlines_by_SIFID_SIFIDHI):
        set_value = start_value + cvalue # combine flags into code
        matchset = index_by_SIFID_pairs(I, C)
        for _, intersection_ids, centerline_ids in matchset.itertuples():
            for intersection_id in intersection_ids:
                for cl_id in centerline_ids:
                    mapping[(intersection_id, cl_id)] |= set_value # set code, preserving flags that are already set
        cvalue = 0b01 # set cvalue flag for next iteration over centerlines_by_SIFID_SIFIDHI

    start_value = 0b0100 # set start_value flag for next iteration over itersections_by_SEC_FST
        
mdf = matchdfnumeric = pd.Series(mapping)

# 
fs= {'fst': lambda x: bool(x & 0b1000),
     'sec': lambda x: bool(x & 0b0100),
     'low': lambda x: bool(x & 0b0010),
     'hi':  lambda x: bool(x & 0b0001)}

match_matrix = mdf.transform(fs).convert_dtypes() # convert the codes to something more easy to use

match_matrix[match_matrix.sum(axis=1) == 2]
match_matrix

fst    sec    low     hi
589991067518338 10250    True  False   True  False
                28655    True  False   True  False
590506463593858 10250    True  False   True  False
                28655    True  False   True  False
409323268213029 26919    True  False   True  False
...                       ...    ...    ...    ...
847134907955010 174788  False   True  False   True
847207302035270 175429  False   True  False   True
847212217628486 175434  False   True  False   True
847246577432390 175435  False   True  False   True
847250872399689 175437  False   True  False   True

[64111 rows x 4 columns]

In [44]:
mapping = defaultdict(int)

fst_mask = 0b0001
sec_mask = 0b0010
low_mask = 0b0100
hi_mask = 0b1000

for imask, I in ((fst_mask, intersections_by_FST_SEC), (sec_mask, intersections_by_SEC_FST)):
    for cmask, C in ((low_mask, centerlines_by_SIFID_SIFIDLOW), (hi_mask, centerlines_by_SIFID_SIFIDHI)):
        matchset = index_by_SIFID_pairs(I, C)
        set_cols = imask + cmask
        for (s1, s2), intersection_ids, centerline_ids in matchset.itertuples():
            for intersection_id in intersection_ids:
                for cl_id in centerline_ids:
                    mapping[(intersection_id, cl_id)] |= set_cols

matchdfnumeric = pd.Series(mapping)

matchdfnumeric.index.set_names(('int_id', 'cl_id'), inplace=True)

In [45]:
iw = matchdfnumeric.index.to_frame()

#int_d = intersections[['FST_ROADNAME', 'SEC_ROADNAME']]
cl_d = centerlines[['ROADNAME', 'SIFIDLOW', 'SIFIDHI']]

def get_int_names(int_id):
    int_d.loc[int_id]

def get_cl_names(cl_id):
    rw, sl, sh = cl_d.loc[cl_id]
    sl = sifid_to_roadname.get(sl)
    sh = sifid_to_roadname.get(sh)
    return pd.Series({
        'ROADNAME': rw,
        'LO_cross': sl,
        'HI_cross': sh})

NAME_df = iw.cl_id.apply(get_cl_names)
#matchdfnumeric.index.to_frame()
#4.9 sec

NAME_df

ROADNAME          LO_cross          HI_cross
int_id          cl_id                                                       
589991067518338 10250            NO NAME      BLUE WING DR              None
                28655            NO NAME      BLUE WING DR              None
590506463593858 10250            NO NAME      BLUE WING DR              None
                28655            NO NAME      BLUE WING DR              None
409323268213029 26919            NO NAME    CAMP GROUND RD              None
...                                  ...               ...               ...
847134907955010 174788   RINGING BELL LN  HALDEN RIDGE WAY  RINGING BELL CIR
847207302035270 175429  COPPER DRIFT WAY  BRICK FORGE PASS     BLACKSMITH RD
847212217628486 175434  COPPER DRIFT WAY              None    PINE BELLOW LN
847246577432390 175435  COPPER DRIFT WAY    PINE BELLOW LN  BRICK FORGE PASS
847250872399689 175437      FIRESCALE CT              None  BRICK FORGE PASS

[64111 rows x 3 columns]

In [46]:
cl_dat = centerlines[['SIFID', 'SIFIDLOW', 'SIFIDHI']]

def gc(ci):
    return cl_dat.loc[ci]

sifm = matchdfnumeric.index.to_frame().cl_id.apply(gc)
sifm


SIFID  SIFIDLOW  SIFIDHI
int_id          cl_id                           
589991067518338 10250       1       591     8594
                28655       1       591     8594
590506463593858 10250       1       591     8594
                28655       1       591     8594
409323268213029 26919       1       897     8594
...                       ...       ...      ...
847134907955010 174788  15593     15254    15594
847207302035270 175429  15597     15599    10281
847212217628486 175434  15597      8594    15598
847246577432390 175435  15597     15598    15599
847250872399689 175437  15600      8594    15599

[64111 rows x 3 columns]

In [47]:

any(matchdfnumeric == 0b0011) # none
any(matchdfnumeric == 0b0000) # false # none of these either

data = {"int_match": matchdfnumeric & 0b0011,
        "cl_sifid": sifm.SIFID,
        "low_match": sifm.SIFIDLOW[(matchdfnumeric & low_mask) != 0],
        "hi_match": sifm.SIFIDHI[(matchdfnumeric & hi_mask) != 0]}


#dd = pd.concat((int_ser, sifm.SIFID, low_ser, hi_ser), axis=1, keys=['int_match', 'cl_sifid', 'low_match', 'hi_match']).convert_dtypes()

SIFdf = pd.DataFrame.from_dict(data).convert_dtypes()

In [48]:
def ix(columns, *labels):
    out = list()
    columns = iter(columns)
    for label, column in zip(labels, columns):
        out.append((label, column))
    else:
        # default = last value of label
        for column in columns:
            out.append((label, column))
    return pd.MultiIndex.from_tuples(out)

ix(NAME_df.columns, 'hello')

NAME_df.columns=ix(NAME_df.columns, 'hello')

In [49]:

euclidean_distance = np.linalg.norm
#

#
intersections_GEO = intersections.GEOMETRY
centerlines_GEOLOW = centerlines.GEOLOW
centerlines_GEOHI = centerlines.GEOHI


In [50]:
#ids = mdf.index.to_series().transform({'intersection_ids':itemgetter(0), 'centerline_ids':itemgetter(1)})
ids = mdf.index.to_frame(name=['intersection_ids',"centerline_ids"])

int_points = ids.intersection_ids.apply(lambda intid: intersections_GEO.at[intid])

hi_points = ids.centerline_ids.apply(lambda cl_id: centerlines_GEOHI.at[cl_id])
hi_dist = (int_points - hi_points).apply(euclidean_distance)

low_points = ids.centerline_ids.apply(lambda cl_id: centerlines_GEOLOW.at[cl_id])
low_dist = (int_points - low_points).apply(euclidean_distance)

def encode_closer_point(difference):
    if difference < 0: # hi_dist < low_dist
        return 'hi'
    elif difference > 0: # hi_dist > low_dist
        return 'low'
    elif difference == 0: # hi_dist == low_dist
        return 'both'
    else:
        return pd.NA

dist_diff = (hi_dist - low_dist)
geo_end = dist_diff.apply(encode_closer_point)


geo_end.hasnans # -> False.
# checking something;
(geo_end[geo_end != 'low'] == geo_end[(geo_end == 'hi') | (geo_end == 'both')]).all() # -> True

np.True_

In [51]:
close_points = pd.concat((
    hi_points[geo_end != 'low'], 
    low_points[geo_end == 'low']))

close_distance = pd.concat((
    hi_dist[geo_end != 'low'],
    low_dist[geo_end == 'low']))

far_points = pd.concat((
    hi_points[geo_end == 'low'],
    low_points[geo_end == 'hi']))

far_distance = pd.concat((
    hi_dist[geo_end == 'low'],
    low_dist[geo_end == 'hi']))


working = pd.DataFrame(pd.Series(int_points, name='int_point'))
working['geo_end'] = geo_end
working['close_point'] = close_points
working['far_point'] = far_points
working['close_dist'] = close_distance
working['far_dist'] = far_distance
working['dist_diff'] = dist_diff.abs()

working.index.set_names(('int_id', 'cl_id'), inplace=True)
geo_data = working

geo_data

int_point geo_end  \
int_id          cl_id                                             
589991067518338 10250   [-85.8920292933, 38.1439149601]     low   
                28655   [-85.8920292933, 38.1439149601]     low   
590506463593858 10250      [-85.892052272, 38.14321909]     low   
                28655      [-85.892052272, 38.14321909]     low   
409323268213029 26919   [-85.8301302406, 38.2136024637]     low   
...                                                 ...     ...   
847134907955010 174788  [-85.4966887661, 38.2906992036]      hi   
847207302035270 175429  [-85.5586236853, 38.1263603351]      hi   
847212217628486 175434  [-85.5588442442, 38.1280189286]      hi   
847246577432390 175435  [-85.5585370685, 38.1271574049]      hi   
847250872399689 175437  [-85.5575279734, 38.1270959347]      hi   

                                                 close_point  \
int_id          cl_id                                          
589991067518338 10250         [-85.8920342109, 38.143922224]   
                28655        [-85.8920571895, 38.1432263537]   
590506463593858 10250         [-85.8920342109, 38.143922224]   
                28655        [-85.8920571895, 38.1432263537]   
409323268213029 26919        [-85.8301351454, 38.2136097438]   
...                                                      ...   
847134907955010 174788       [-85.4966935792, 38.2907065141]   
847207302035270 175429       [-85.5586285042, 38.1263676111]   
847212217628486 175434       [-85.5588490633, 38.1280262049]   
847246577432390 175435  [-85.5585418875, 38.127164681000004]   
847250872399689 175437       [-85.5575327922, 38.1271032109]   

                                                   far_point  close_dist  \
int_id          cl_id                                                      
589991067518338 10250        [-85.8901779074, 38.1438753086]    0.000009   
                28655        [-85.8953593354, 38.1433191649]    0.000689   
590506463593858 10250        [-85.8901779074, 38.1438753086]    0.000703   
                28655        [-85.8953593354, 38.1433191649]    0.000009   
409323268213029 26919        [-85.8320107851, 38.2120229119]    0.000009   
...                                                      ...         ...   
847134907955010 174788       [-85.4975897985, 38.2902281308]    0.000009   
847207302035270 175429  [-85.5585418875, 38.127164681000004]    0.000009   
847212217628486 175434       [-85.5589900382, 38.1289572117]    0.000009   
847246577432390 175435       [-85.5588490633, 38.1280262049]    0.000009   
847250872399689 175437       [-85.5566407119, 38.1273321786]    0.000009   

                        far_dist  dist_diff  
int_id          cl_id                        
589991067518338 10250   0.001852   0.001843  
                28655   0.003383   0.002694  
590506463593858 10250   0.001986   0.001283  
                28655   0.003309   0.003300  
409323268213029 26919   0.002456   0.002447  
...                          ...        ...  
847134907955010 174788  0.001017   0.001008  
847207302035270 175429  0.000808   0.000800  
847212217628486 175434  0.000950   0.000941  
847246577432390 175435  0.000923   0.000914  
847250872399689 175437  0.000918   0.000909  

[64111 rows x 7 columns]

In [52]:

def get_intersection_names(intersection_id):
    in1 = intersections.at[intersection_id, 'FST_ROADNAME']
    in2 = intersections.at[intersection_id, 'SEC_ROADNAME']
    return in1, in2

names = centerlines.groupby("SIFID").ROADNAME.apply(set)#.apply(lambda x:len(x)==1).all() # true
sifid_names = names.apply(lambda x:x.pop())

def roadname_by_sifid(sifid):
    return sifid_names[sifid]

def get_names(int_id, cd_id):
    inames = get_intersection_names(intersection_id)
    cnames = roadname_by_sifid(cl_id)
    return {'intersection':inames, 'centerline':cnames}



In [53]:
intids = working.index.to_frame().int_id
intids

int_id           cl_id 
589991067518338  10250     589991067518338
                 28655     589991067518338
590506463593858  10250     590506463593858
                 28655     590506463593858
409323268213029  26919     409323268213029
                                ...       
847134907955010  174788    847134907955010
847207302035270  175429    847207302035270
847212217628486  175434    847212217628486
847246577432390  175435    847246577432390
847250872399689  175437    847250872399689
Name: int_id, Length: 64111, dtype: int64

In [54]:
working = geo_data[geo_data.close_dist >= .1]
working

iw = working.index.to_frame()
iw

def get_int_data(int_id):
    return intersections.loc[int_id][["FST_ROADNAME", "FST_SIFID", "SEC_ROADNAME", "SEC_SIFID"]]

def get_cl_data(cl_id):
    return centerlines.loc[cl_id][['ROADNAME', 'SIFID', 'CORE_CLASS']]

data = pd.concat((
iw.cl_id.transform({"centerline_data":get_cl_data}) ,
iw.int_id.transform({'intersection_data':get_int_data})
), axis=1)

data


def mmft(label, df):
    if isinstance(label, str):
        return pd.MultiIndex.from_tuples((label, col) for col in df.columns)
    else:
        return pd.MultiIndex.from_tuples((L, C) for L, C in zip(label, df.columns))
    


def dd(geodf):
    #ex = geodf[['int_point
    #out = list()
    iw = geodf.index.to_frame()
    cl_geo = geodf[['int_point', 'close_point', 'close_dist', 'geo_end']]
    cl_geo.columns = mmft(('int_geo', 'cl_geo', 'cl_geo', 'cl_geo'), cl_geo)

    cl_data = iw.cl_id.apply(get_cl_data)
    cl_data.columns = mmft('cl_data', cl_data)
    # TODO get hi/low sifid / roadname

    int_data = iw.int_id.apply(get_int_data)
    int_data.columns = mmft("int_data", int_data)

    out = [int_data, cl_geo, cl_data]
    return pd.concat(out, axis=1)
    
    int_data = iw.int_id.apply(get_int_data)
    int_info = pd.concat((int_data, geodf.int_point), axis=1)
    int_info.columns = pd.MultiIndex.from_tuples(('int_data', col) for col in int_info.columns)
    
    return pd.concat((int_info, cl_info), axis=1).sort_index()

#    pd.MultiIndex.from_tuples([('cl_data', col) for col in c.columns])
#c.columns = pd.MultiIndex.from_tuples([('cl_data', col) for col in c.columns])
#c

see = dd(working)

#see[~see.int_data.FST_ROADNAME.str.contains('841')]

see

int_data                                    \
                          FST_ROADNAME FST_SIFID  SEC_ROADNAME SEC_SIFID   
int_id           cl_id                                                     
590467808889140  29463  NO STREET NAME         1   CANE RUN RD       906   
38521867941168   6218     COLUMBIA AVE      1245  KENTUCKY AVE      3356   
589634891494704  21710    COLUMBIA AVE      1245  KENTUCKY AVE      3356   
706470196341142  903       KY-841 RAMP     14021        KY 841     14195   
                 18413     KY-841 RAMP     14021        KY 841     14195   
...                                ...       ...           ...       ...   
654251992213784  6590           KY 841     14195   KY-841 RAMP     14021   
                 7706           KY 841     14195   KY-841 RAMP     14021   
2696491761592600 6590           KY 841     14195   KY-841 RAMP     14021   
                 7706           KY 841     14195   KY-841 RAMP     14021   
                 11399          KY 841     14195   KY-841 RAMP     14021   

                                                int_geo  \
                                              int_point   
int_id           cl_id                                    
590467808889140  29463  [-85.8967398176, 38.1433580312]   
38521867941168   6218   [-85.6184455169, 38.2600006953]   
589634891494704  21710  [-85.8480563282, 38.1508005158]   
706470196341142  903    [-85.7024627213, 38.1152264972]   
                 18413  [-85.7024627213, 38.1152264972]   
...                                                 ...   
654251992213784  6590    [-85.8700263878, 38.091373397]   
                 7706    [-85.8700263878, 38.091373397]   
2696491761592600 6590   [-85.8769126885, 38.0929399762]   
                 7706   [-85.8769126885, 38.0929399762]   
                 11399  [-85.8769126885, 38.0929399762]   

                                                 cl_geo                     \
                                            close_point close_dist geo_end   
int_id           cl_id                                                       
590467808889140  29463  [-85.8175947344, 38.2187237448]   0.109288      hi   
38521867941168   6218    [-85.848070012, 38.1508080266]   0.254265     low   
589634891494704  21710  [-85.6184503633, 38.2600079942]   0.254254     low   
706470196341142  903    [-85.8189716474, 38.1032602249]   0.117122     low   
                 18413  [-85.8700312951, 38.0913806517]   0.169257     low   
...                                                 ...        ...     ...   
654251992213784  6590   [-85.7547983165, 38.1174115631]   0.118133      hi   
                 7706   [-85.7479769634, 38.1166977252]   0.124649     low   
2696491761592600 6590   [-85.7547983165, 38.1174115631]   0.124542      hi   
                 7706   [-85.7479769634, 38.1166977252]   0.131106     low   
                 11399   [-85.7754995491, 38.119467301]   0.104825     low   

                             cl_data                         
                            ROADNAME  SIFID      CORE_CLASS  
int_id           cl_id                                       
590467808889140  29463       NO NAME      1           LOCAL  
38521867941168   6218   COLUMBIA AVE   1245           LOCAL  
589634891494704  21710  COLUMBIA AVE   1245           LOCAL  
706470196341142  903     KY-841 RAMP  14021  MAJOR ARTERIAL  
                 18413   KY-841 RAMP  14021  MAJOR ARTERIAL  
...                              ...    ...             ...  
654251992213784  6590    KY-841 RAMP  14021  MAJOR ARTERIAL  
                 7706    KY-841 RAMP  14021  MAJOR ARTERIAL  
2696491761592600 6590    KY-841 RAMP  14021  MAJOR ARTERIAL  
                 7706    KY-841 RAMP  14021  MAJOR ARTERIAL  
                 11399   KY-841 RAMP  14021  MAJOR ARTERIAL  

[89 rows x 11 columns]

In [55]:
iw = working.index.to_frame()

def get_mm(iw):
    cl_id = iw.cl_id
    cl = centerlines.loc[cl_id]
    int_id = iw.int_id
    m = match_matrix.loc[int_id]
    if m.FST:
        ...

    cl_lowsif = cl_hisif = None
    if m.low:
        cl_lowsif = cl.SIFIDLOW
    if m.hi:
        cl_hisif = cl.SIFIDHI

def f(row):
    return type(row)

match_matrix

fst    sec    low     hi
589991067518338 10250    True  False   True  False
                28655    True  False   True  False
590506463593858 10250    True  False   True  False
                28655    True  False   True  False
409323268213029 26919    True  False   True  False
...                       ...    ...    ...    ...
847134907955010 174788  False   True  False   True
847207302035270 175429  False   True  False   True
847212217628486 175434  False   True  False   True
847246577432390 175435  False   True  False   True
847250872399689 175437  False   True  False   True

[64111 rows x 4 columns]

In [56]:
match_matrix.loc[847212217628486, 175434].loc[['low', 'hi']]

low    False
hi      True
Name: (847212217628486, 175434), dtype: boolean

In [57]:

c = iw.cl_id.apply(get_cl_data)
i = iw.int_id.apply(get_int_data)


df=pd.DataFrame({'a':[1,2,3],'b':[4,5,6]})

columns=[('c','a'),('c','b')]

df.columns=pd.MultiIndex.from_tuples(columns)
df

pd.MultiIndex.from_tuples([('cl_data', col) for col in c.columns])
c.columns = pd.MultiIndex.from_tuples([('cl_data', col) for col in c.columns])
c

cl_data                       
                            ROADNAME  SIFID      CORE_CLASS
int_id           cl_id                                     
590467808889140  29463       NO NAME      1           LOCAL
38521867941168   6218   COLUMBIA AVE   1245           LOCAL
589634891494704  21710  COLUMBIA AVE   1245           LOCAL
706470196341142  903     KY-841 RAMP  14021  MAJOR ARTERIAL
                 18413   KY-841 RAMP  14021  MAJOR ARTERIAL
...                              ...    ...             ...
654251992213784  6590    KY-841 RAMP  14021  MAJOR ARTERIAL
                 7706    KY-841 RAMP  14021  MAJOR ARTERIAL
2696491761592600 6590    KY-841 RAMP  14021  MAJOR ARTERIAL
                 7706    KY-841 RAMP  14021  MAJOR ARTERIAL
                 11399   KY-841 RAMP  14021  MAJOR ARTERIAL

[89 rows x 3 columns]

In [58]:
s= working.index.get_level_values('cl_id')

centerlines.loc[s].ROADNAME.value_counts()

ROADNAME
KY-841 RAMP     39
KY 841          36
COLUMBIA AVE     4
CANE RUN RD      4
KENTUCKY AVE     4
NO NAME          2
Name: count, dtype: int64

In [59]:
mi = intersections.loc[working.index.get_level_values('int_id')]

# KY 841 / I 265 / Watterson Expwy and ramps
mi[(mi.FST_ROADNAME.str.contains("841") & mi.SEC_ROADNAME.str.contains("841"))]

# other weird matches
wm = mi[~(mi.FST_ROADNAME.str.contains("841") & mi.SEC_ROADNAME.str.contains("841"))]

working.loc[wm.index, :]
wm
mi


,FST_ROADNAME,FST_SIFID,SEC_ROADNAME,SEC_SIFID,GEOMETRY
int_id,,,,,
590467808889140,NO STREET NAME,1,CANE RUN RD,906,"[-85.8967398176, 38.1433580312]"
38521867941168,COLUMBIA AVE,1245,KENTUCKY AVE,3356,"[-85.6184455169, 38.2600006953]"
589634891494704,COLUMBIA AVE,1245,KENTUCKY AVE,3356,"[-85.8480563282, 38.1508005158]"
706470196341142,KY-841 RAMP,14021,KY 841,14195,"[-85.7024627213, 38.1152264972]"
706470196341142,KY-841 RAMP,14021,KY 841,14195,"[-85.7024627213, 38.1152264972]"
...,...,...,...,...,...
654251992213784,KY 841,14195,KY-841 RAMP,14021,"[-85.8700263878, 38.091373397]"
654251992213784,KY 841,14195,KY-841 RAMP,14021,"[-85.8700263878, 38.091373397]"
2696491761592600,KY 841,14195,KY-841 RAMP,14021,"[-85.8769126885, 38.0929399762]"


In [60]:


    

#display(
#info[['int_point', 'close_point', 'close_dist', 'geo_end', ]])



wmm = data[~data.intersection_data.FST_ROADNAME.str.contains('841')]

display(
working.loc[wmm.index][['int_point', 'close_point', 'close_dist', 'geo_end', ]],
wmm)


,,int_point,close_point,close_dist,geo_end
int_id,cl_id,,,,
590467808889140,29463,"[-85.8967398176, 38.1433580312]","[-85.8175947344, 38.2187237448]",0.109288,hi
38521867941168,6218,"[-85.6184455169, 38.2600006953]","[-85.848070012, 38.1508080266]",0.254265,low
589634891494704,21710,"[-85.8480563282, 38.1508005158]","[-85.6184503633, 38.2600079942]",0.254254,low
352187318274356,31519,"[-85.8150265884, 38.2184243755]","[-85.8953593354, 38.1433191649]",0.109973,low
38521867941168,10508,"[-85.6184455169, 38.2600006953]","[-85.8462075197, 38.1506974839]",0.252632,low
589634891494704,11695,"[-85.8480563282, 38.1508005158]","[-85.6195336226, 38.2594089616]",0.253019,low
352187318274356,11220,"[-85.8150265884, 38.2184243755]","[-85.8967911478, 38.1433697959]",0.110989,low
590467808889140,26722,"[-85.8967398176, 38.1433580312]","[-85.8179721092, 38.2154415187]",0.106773,hi
38521867941168,2474,"[-85.6184455169, 38.2600006953]","[-85.848070012, 38.1508080266]",0.254265,low


centerline_data                           \
                             ROADNAME SIFID         CORE_CLASS   
int_id          cl_id                                            
590467808889140 29463         NO NAME     1              LOCAL   
38521867941168  6218     COLUMBIA AVE  1245              LOCAL   
589634891494704 21710    COLUMBIA AVE  1245              LOCAL   
352187318274356 31519         NO NAME     1              LOCAL   
38521867941168  10508    COLUMBIA AVE  1245              LOCAL   
589634891494704 11695    COLUMBIA AVE  1245              LOCAL   
352187318274356 11220     CANE RUN RD   906  PRIMARY COLLECTOR   
590467808889140 26722     CANE RUN RD   906     MINOR ARTERIAL   
38521867941168  2474     KENTUCKY AVE  3356              LOCAL   
589634891494704 29055    KENTUCKY AVE  3356              LOCAL   
352187318274356 32589     CANE RUN RD   906  PRIMARY COLLECTOR   
590467808889140 5361      CANE RUN RD   906     MINOR ARTERIAL   
38521867941168  10465    KENTUCKY AVE  3356              LOCAL   
589634891494704 5523     KENTUCKY AVE  3356              LOCAL   

                      intersection_data                                    
                           FST_ROADNAME FST_SIFID  SEC_ROADNAME SEC_SIFID  
int_id          cl_id                                                      
590467808889140 29463    NO STREET NAME         1   CANE RUN RD       906  
38521867941168  6218       COLUMBIA AVE      1245  KENTUCKY AVE      3356  
589634891494704 21710      COLUMBIA AVE      1245  KENTUCKY AVE      3356  
352187318274356 31519    NO STREET NAME         1   CANE RUN RD       906  
38521867941168  10508      COLUMBIA AVE      1245  KENTUCKY AVE      3356  
589634891494704 11695      COLUMBIA AVE      1245  KENTUCKY AVE      3356  
352187318274356 11220    NO STREET NAME         1   CANE RUN RD       906  
590467808889140 26722    NO STREET NAME         1   CANE RUN RD       906  
38521867941168  2474       COLUMBIA AVE      1245  KENTUCKY AVE      3356  
589634891494704 29055      COLUMBIA AVE      1245  KENTUCKY AVE      3356  
352187318274356 32589    NO STREET NAME         1   CANE RUN RD       906  
590467808889140 5361     NO STREET NAME         1   CANE RUN RD       906  
38521867941168  10465      COLUMBIA AVE      1245  KENTUCKY AVE      3356  
589634891494704 5523       COLUMBIA AVE      1245  KENTUCKY AVE      3356

In [61]:
centerlines[centerlines.ROADNAME.str.contains("KENTUCKY AVE")]
centerlines[(centerlines.SIFID == 3356) & ((centerlines.SIFIDLOW == 1245) | (centerlines.SIFIDHI == 1245))]

#wm = wm.groupby(by="FST_SIFID").apply(lambda x:x)


,ROADNAME,SIFID,SIFIDLOW,SIFIDHI,GEOLOW,GEOHI,CORE_CLASS
2474,KENTUCKY AVE,3356,1245,4725,"[-85.848070012, 38.1508080266]","[-85.848221191, 38.1490259808]",LOCAL
5523,KENTUCKY AVE,3356,2249,1245,"[-85.6148949649, 38.2557432145]","[-85.6184503633, 38.2600079942]",LOCAL
10465,KENTUCKY AVE,3356,8594,1245,"[-85.8478987894, 38.1526162695]","[-85.848070012, 38.1508080266]",LOCAL
29055,KENTUCKY AVE,3356,1245,8594,"[-85.6184503633, 38.2600079942]","[-85.620123247, 38.2599229012]",LOCAL


In [62]:
def convert_geo(geo):
    if len(geo) == 2:
        long, lat = geo
        return (lat, long)
    else:
        return [(lat, long) for long, lat in geo]

def swap_point(point):
    x, y = point
    return (y, x)

swap_point((1,2))
        

(2, 1)

In [63]:
# working = mdf.index.to_series()


# def get_geo(row):
#     int_id, cl_id = row
#     out = dict() # tried Series. Took too long. Dict is very fast ( .4 sec)
#     out['intx_geo'] = intx_geo = intersections_GEO.at[int_id]
#     out['cl_geo_low'] = centerlines_GEOLOW.at[cl_id]
#     out['cl_geo_hi'] = centerlines_GEOHI.at[cl_id]
#     return out

# working = pd.DataFrame(mdf.index.to_series().apply(get_geo).to_list(), index=mdf.index)
# hi_dist = (working.intx_geo - working.cl_geo_hi).apply(euclidean_distance)
# low_dist = (working.intx_geo - working.cl_geo_low).apply(euclidean_distance)

# def find_close_closer_point(x):
#     if x < 0:
#         return 'hi'
#     elif x > 0:
#         return 'low'
#     elif x == 0:
#         return 'both'
#     else:
#         return pd.NA

# working['geo_end'] = (hi_dist - low_dist).apply(find_close_closer_point)
# working

# def gp(row):
#     code = row.geo_end
#     if code == 'low':
#         return (row.cl_geo_low, row.cl_geo_hi)
#     elif code == 'hi':
#         return (row.cl_geo_hi,row.cl_geo_low)
#     elif code == 'both':
#         return (row.cl_geo_hi, None)
#     else:
#         return (None, None)

# ee = pd.DataFrame(working.apply(gp, axis=1).to_list(), index=working.index, columns=['close_point', 'far_point'])
# pd.concat((working, ee), axis=1)


In [64]:
# old versions of code
#     lowdist = euclidean_distance(intx_geo - cl_geo_low)
#     hidist = euclidean_distance(intx_geo - cl_geo_hi)
#     if lowdist < hidist:
#         geo_close = 'low'
#         closer_point = cl_geo_low
#         farther_point = cl_geo_hi
#         close_dist = lowdist
#         far_dist = hidist
#     elif lowdist > hidist:
#         geo_close = 'hi'
#         closer_point = cl_geo_hi
#         farther_point = cl_geo_low
#         close_dist = lowdist
#         far_dist = hidist
#     elif lowdist == hidist:
#         geo_close = 'both'
#         #assert cl_geo_low == cl_geo_hi
#         closer_point = cl_geo_low
#         farther_point = pd.NA
#         close_dist = lowdist
#         far_dist = hidist

#     out['geo_close'] = geo_close
#     out['close_point'] = closer_point
#     out['far_point'] = farther_point
#     out['close_dist'] = close_dist
#     out['far_dist'] = far_dist

#     return out


# working = pd.DataFrame(mdf.index.to_series().apply(find_close_closer_point).to_list(), index=mdf.index)
# #centerlines.loc[working[working.closer_code == 'both'].index.get_level_values(1)] # closer_code = 'both'


In [65]:

diffs = (working['close_dist'] - working['far_dist']).abs()
# some values are zero
# drop these to make finding min easier
dd = diffs.drop(diffs[diffs == 0].index)


dd.min()

np.float64(0.0005881446927083755)

In [66]:
# Problematic numbers?

#intersections.loc[319438198302071]
#centerlines.loc[22602]

#centerlines.loc[[84838, 31468]]
#(-85.5284187234, 38.2003603491)
#intersections.loc[[14300767569, 10005800273]]
# Problematic numbers?


#centerlines.loc[[13164, 24805, 26558]],
#intersections.loc[389674641274662])

#low_match_1[low_match_1 == 389674641274662]

# cl, ix =88673, {334354632500584, 726302145904921}


# 5 have 3 matches

# removing ramps from centerlines removed 2 of these

# 11577    {318119662229912, 318128252164504, 31811536726...
# Arthur St. and I 65 RAMP

# 79773    {731803473713560, 844908543362584, 84487847859...
# I 65 ramp

# 80073    {844438991051160, 731813464524312, 73181635861...
# I 65 ramp

# 31660    {582161378284545, 582191443055617, 61834218278...

#582161378284545	AUTUMN WAY	244	SUMMERTIME PKY	8232	(-85.8735529704, 38.0723544119)
#582191443055617	AUTUMN WAY	244	SUMMERTIME PKY	8232	(-85.8741780941, 38.0721878176)
#618342182786049	AUTUMN WAY	244	SUMMERTIME PKY	8232	(-85.873007987, 38.0733107806)

# 19920    {731744745003415, 722828392896919, 72287563753...
# I 65 RAMP



# 1 has 4 matches -> 
# removing EXPRESSWAYS from centerlines dealt with this one.

# 79772 : {731803473713560, 731813464524312, 844878478591512, 844908543362584}
# I 65 NORTH x I 65 RAMP
# no more from there 


# potential problematic numbers

#set(hi_match.keys()).intersection(hi_match2.keys()) # empty : good news
#  14288: ('start', 360168983520312, 8.767631726955189e-06),

#display(
##centerlines.loc[14288],
#intersections.loc[[360168983520312, 360748804105272]]
#

#centerlines.loc[[52803, 20860]]
#centerlines[centerlines.SIFIDLOW == 8594]
#centerlines.loc[[ 1498,  2541,  2610,  3823,  3882,  5039,  6722,  7769,  8692, 11961,
#       14175, 15423, 19704, 23933, 24872, 27737, 29258, 30870]] # I 265 RAMP.
#intersections.loc[847265632743817]
#intersections.loc[722828392896919]
#centerlines.loc[19920]
#ICpairs.loc[6146]

In [67]:
# roadways = centerlines.groupby('SIFID').groups
# intxn_by_fst_sifid = intersections.groupby('FST_SIFID').groups
# # intxn_by_sec_sifid = intersections.groupby('SEC_SIFID') # probably not necessary
#     # ... since all records will be accessed by iterating over the fst_sifid groupby

# #intersections.FST_SIFID.hasnans # == False this is good

# def map_intersection_ids_to_centerline_ids(intersection_sifid_groups=intxn_by_fst_sifid, centerline_sifid_groups=roadways):
#     found = dict()
#     notfound = list()
#     for sifid_1 in intersection_sifid_groups.keys():
#         roadway_ids = centerline_sifid_groups.get(sifid_1, None)
#         if roadway_ids is not None:
#             found[sifid_1] = roadway_ids # first sifid match to centerlines id
#         else:
#             notfound.append(sifid_1)
#     return found, notfound

# def map_intersections_to_centerlines(intersections=intersections, centerlines=centerlines):
#     mapping = pd.DataFrame(columns=['full_match', 'fst_match_only', 'sec_match_only'])
#     full_matches = pd.Series(name='full_match', dtype="O")
#     roadways = centerlines.groupby('SIFID').groups
#     intersections_by_first_sifid = intersections.groupby('FST_SIFID').groups
#     notfound = list()

#     for sifid_1, intersection_ids in intersections_by_first_sifid.items():
#         centerline_ids = roadways.get(sifid_1, None)
#         if centerline_ids is None:
#             notfound.extend(intersection_ids)
#         else:
#             fst_match = centerlines.loc[centerline_ids]
#             for intersection_id in intersection_ids:
#                 sifid_2 = intersections.at[intersection_id, 'SEC_SIFID']
#                 full_match = fst_match[(fst_match.SIFIDLOW == sifid_2) | (fst_match.SIFIDHI == sifid_2)]
#                 if full_match.empty:
#                     mapping.at[intersection_id, 'fst_match_only'] = True
#                 else:
#                     full_match = full_match.index.tolist()
#                     #print(full_match)
#                     full_matches.at[intersection_id] = full_match
        
#     if notfound:
#         centerline_sifids = roadways.keys()
#         for intersection_id, sifid_2 in intersections.loc[notfound]['SEC_SIFID'].items():
#             if sifid_2 in centerline_sifids:
#                 mapping.at[intersection_id, 'sec_match_only'] = True

#     return pd.concat((full_matches, mapping))


# mapping = map_intersections_to_centerlines()


In [68]:
#mapping.isna().apply(any, axis=1).all()
# test that each row has at least one value filled -> Yes
#intersections.index.difference(mapping.index) # == Index([693211604576105], dtype='int64')

#display(intersections.loc[693211604576105]) -> 13554, 13555 fst, sec ids

#centerlines[centerlines.SIFID == 13555] # nope, nor 13554 
# probably just ignore this.

#mapping[mapping.full_match.notna()]
#mapping

In [69]:
# mapping = pd.DataFrame(columns=['full_match', 'fst_match_only', 'sec_match_only'])
# roadways = centerlines.groupby('SIFID').groups
# intersections_by_first_sifid = intersections.groupby('FST_SIFID').groups
# notfound = list()

# for sifid_1, intersection_ids in intersections_by_first_sifid.items():
#     centerline_ids = roadways.get(sifid_1, None)
#     if centerline_ids is None:
#         notfound.extend(intersection_ids)
#     else:
#         fst_match = centerlines.loc[centerline_ids]
#         #print(fst_match)
#         for intersection_id in intersection_ids:
#             sifid_2 = intersections.at[intersection_id, 'SEC_SIFID']
#             full_match = fst_match[(fst_match.SIFIDLOW == sifid_2) | (fst_match.SIFIDHI == sifid_2)]



In [70]:
centerlines[centerlines.SIFID==1178]
notfound = [624,626,689,903,2053,122932,139877,161371,161372,168074]

intersections[intersections.FST_SIFID == 1178]
nf1 = intersections[intersections.FST_SIFID.isin(notfound)]
nf2 = centerlines[centerlines.SIFID.isin(nf1.SEC_SIFID.values)] # looks to be all interstate ramps
# it makes sense that these would not be in the centerline data b/c I probably stripped them out at a some point
# you can't/shouldn't ride a bicycle on the ramps/interstate!

noramp = nf2[nf2.CORE_CLASS != "INTERSTATE RAMP"] # 205 ramps. We don't need these roadways.
#display(noramp)
#nf.groupby('CORE_CLASS').count()

#nf.groupby('SIFID').groups
#sifid_twos = intersections[intersections.FST_SIFID.isin(notfound)].SEC_SIFID.values#.groupby('SEC_SIFID').groups
#for sifid_2 in sifid_twos:
#    i = centerlines[centerlines.SIFID == sifid_2].index.tolist()
#    if not i:
#        print(sifid_2)

display(nf1, nf2)

,FST_ROADNAME,FST_SIFID,SEC_ROADNAME,SEC_SIFID,GEOMETRY
INTID,,,,,
25314647488608,BRITTANY VALLEY RD,689,LIME KILN LN,3636,"[-85.6373935734, 38.2990193474]"
25533690815846,BRITTANY VALLEY RD,689,GLENVIEW AVE,2525,"[-85.6434704537, 38.2964091193]"
285078996590724,COFFEE TREE LN,2053,COFFEE TREE PL,2101,"[-85.5369278673, 38.1819592614]"
374478301589619,BOWLES AVE,626,WEBSTER ST,6361,"[-85.7267989454, 38.2571141241]"
374942158031000,BOWLES AVE,626,CABEL ST,875,"[-85.7283190618, 38.2564620304]"
414945534616712,CANDOR AVE,903,GARRS LN,2460,"[-85.8163622937, 38.1897925887]"
422723720394881,CANDOR AVE,903,LINHERK AVE,3654,"[-85.8169944951, 38.1869962274]"
432752417703459,BOWIE CT,624,BOWIE DR,625,"[-85.7026014005, 38.1465724184]"


,ROADNAME,SIFID,SIFIDLOW,SIFIDHI,GEOLOW,GEOHI,CORE_CLASS
3034,LIME KILN LN,3636,3536,5040,"[-85.6447336957, 38.3128469693]","[-85.6455583868, 38.313566956]",PRIMARY COLLECTOR
3125,WEBSTER ST,6361,13416,626,"[-85.726472992, 38.2566529689]","[-85.7268038233, 38.2571214174]",LOCAL
3474,GARRS LN,2460,903,4265,"[-85.8163671926, 38.1897998649]","[-85.8174152907, 38.1898710161]",LOCAL
4249,GLENVIEW AVE,2525,1696,8594,"[-85.6420143435, 38.2943749671]","[-85.6421980563, 38.2945737706]",LOCAL
4476,BOWIE DR,625,624,13326,"[-85.702606263, 38.1465796916]","[-85.7016940176, 38.1468951256]",LOCAL
...,...,...,...,...,...,...,...
31463,GLENVIEW AVE,2525,2626,12896,"[-85.6407907198, 38.2928495414]","[-85.6412544794, 38.2934332755]",LOCAL
32265,COFFEE TREE PL,2101,8594,2053,"[-85.5379593471, 38.1819082319]","[-85.5369458175, 38.1819669912]",LOCAL
32380,GARRS LN,2460,3661,4085,"[-85.8131860153, 38.1896095928]","[-85.8150640388, 38.1897210539]",LOCAL
32411,LIME KILN LN,3636,1197,2517,"[-85.634665912, 38.2955398449]","[-85.6357147464, 38.2968583401]",PRIMARY COLLECTOR


In [71]:
# # # This is where Rehl Rd intersects Tucker station



#ee = centerlines[A][["SIFIDLOW", "SIFIDHI"]].copy()
# 5908 another test

#ee['next'] = pd.Series(dtype='int64')
#ee['previouos'] = pd.Series(dtype='int64')

#indexes = set(ee.index)

#while indexes:
#    row = ee.loc[indexes.pop()]
#    if row.SIFIDHI == row.SIFIDLOW:
#        continue # handle loops: 

""" SIFIDLOW	SIFIDHI	next	previouos
7109	5908	5908	NaN	NaN # next previous should be 12697?
9233	5908	12697	NaN	NaN
...
52804	12697	4674	NaN	NaN

This is where Rehl Rd intersects Tucker station

       | Tucker Station
       |
--Rehl--====Rehl====----Rehl--
                  |
                  | Tucker Station

This makes it too annoying to connect the segments using only SIFIDS. Lots of unusual cases.
We have exact geometry data, just use that
code for this already in centerline_data notebook """

' SIFIDLOW\tSIFIDHI\tnext\tpreviouos\n7109\t5908\t5908\tNaN\tNaN # next previous should be 12697?\n9233\t5908\t12697\tNaN\tNaN\n...\n52804\t12697\t4674\tNaN\tNaN\n\nThis is where Rehl Rd intersects Tucker station\n\n       | Tucker Station\n       |\n--Rehl--====Rehl====----Rehl--\n                  |\n                  | Tucker Station\n\nThis makes it too annoying to connect the segments using only SIFIDS. Lots of unusual cases.\nWe have exact geometry data, just use that\ncode for this already in centerline_data notebook '